In [ ]:
# Frontier models: prompt shoving ---> modeling ---> evaluate models 
# Imports
import os
import sys
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from litellm import completion
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate

In [2]:
# Load environment variables in a file called .env
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
# Check the keys
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")  

OpenAI API Key exists and begins sk-proj-


In [3]:
# Connect to OpenAI client library
openai = OpenAI()

In [4]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# Download the batches dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [6]:
# Invistigate the dataset
columns = ['title', 
'category', 
'price',
'full',
'weight',
'summary',
'prompt',
'id'
]
for i in columns:
    # Combine item with its attrs
    attr = getattr(items[0], i)
    print(attr)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
all beauty
6.99
None
0.0
Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.
None
None


In [7]:
# Prompt shoving
def prompt(item):
    # Don't put any constaints just generalize the respond
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [8]:
# Insvistigate summary feature of single datapoint
print(test[0].summary)

Title: Vzaahu Silicone Ice Cube Trays with Lid - 2 Pack  
Category: Kitchen & Dining  
Brand: Vzaahu  
Description: Durable, flexible silicone ice cube trays with lids for easy release and odor prevention.  
Details: Features food-grade silicone, stackable design, removable spill-resistant lid, dishwasher safe, and customizable with fruits or flavors.


In [9]:
prompt(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Vzaahu Silicone Ice Cube Trays with Lid - 2 Pack  \nCategory: Kitchen & Dining  \nBrand: Vzaahu  \nDescription: Durable, flexible silicone ice cube trays with lids for easy release and odor prevention.  \nDetails: Features food-grade silicone, stackable design, removable spill-resistant lid, dishwasher safe, and customizable with fruits or flavors.'}]

In [10]:
# Test dataset
def gpt_4_1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=prompt(item))
    return response.choices[0].message.content

In [11]:
# Test single data point
gpt_4_1_nano(test[0])

'$15.99'

In [12]:
# Evaluation
evaluate(gpt_4_1_nano, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$7 $30 $6 $3 $2 $12 $4 $6 $4 $4 $10 $6 $16 $45 $4 $9 $22 $3 $150 $38 $5 $6 $6 $14 $7 $32 $0 $24 $1 $1 $15 $5 $30 $76 $12 $29 $1 $2 $14 $0 $2 $35 $12 $18 $20 $2 $25 $6 $2 $5 $8 $7 $0 $1 $9 $6 $35 $19 $7 $8 $15 $0 $17 $2 $24 $55 $6 $5 $20 $5 $3 $0 $7 $19 $30 $11 $0 $83 $15 $175 $0 $2 $8 $10 $80 $10 $40 $12 $9 $11 $0 $1 $5 $5 $31 $12 $3 $2 $2 $0 $0 $8 $13 $4 $25 $12 $2 $20 $3 $10 $6 $5 $250 $30 $30 $9 $10 $29 $8 $2 $2 $12 $3 $0 $125 $6 $31 $13 $17 $81 $5 $9 $52 $60 $10 $2 $5 $2 $14 $6 $1 $0 $28 $3 $1 $6 $5 $24 $4 $7 $30 $5 $14 $11 $20 $10 $3 $2 $230 $5 $5 $20 $4 $32 $58 $932 $15 $4 $50 $2 $6 $11 $25 $75 $10 $11 $3 $1 $1 $0 $20 $2 $0 $4 $6 $12 $5 $14 $4 $16 $3 $5 $5 $17 $8 $0 $40 $1 $23 $35 $0 $1 $15 $10 $10 $3 $15 $50 $6 $1 $100 $7 $28 $9 $12 $100 $5 $9 $100 $30 $29 $2 $17 $5 $12 $15 $13 $17 $13 $10 $14 $16 $26 $5 $2 $22 $35 $22 $10 $2 $84 $6 $13 $11 $1 $4 $1 $11 $80 $390 $14 $11 $18 $4 $25 $224 $20 $6 $175 $25 $1 $9 $12 $7 $1 $5 $3 $17 $57 $36 $10 $50 $3 $9 $3 $2 $34 $25 $28 $3 $7 $16 $7

In [13]:
# Test dataset
def gpt_5_1(item):
    response = completion(model="gpt-5.1", messages=prompt(item), reasoning_effort='high')
    return response.choices[0].message.content

In [14]:
# Test single data point
gpt_5_1(test[0])

'$9.99'

In [15]:
# Evaluation
evaluate(gpt_5_1, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$1 $10 $0 $4 $8 $9 $2 $7 $1 $4 $13 $1 $3 $5 $1 $1 $4 $2 $110 $68 $3 $1 $4 $1 $2 $1 $0 $14 $0 $6 $6 $2 $10 $274 $2 $24 $6 $3 $14 $6 $3 $20 $4 $18 $5 $1 $10 $1 $1 $5 $2 $7 $2 $2 $0 $3 $35 $19 $1 $1 $15 $0 $4 $4 $9 $10 $2 $8 $5 $5 $0 $1 $2 $49 $5 $6 $0 $83 $10 $170 $0 $7 $6 $10 $30 $5 $10 $2 $9 $4 $2 $4 $3 $3 $6 $0 $2 $2 $3 $0 $3 $3 $10 $1 $20 $0 $2 $10 $17 $0 $11 $5 $30 $10 $0 $2 $1 $79 $3 $12 $17 $7 $1 $4 $25 $18 $26 $1 $7 $52 $0 $2 $152 $0 $5 $6 $4 $0 $6 $4 $0 $20 $3 $3 $5 $1 $3 $24 $6 $1 $10 $20 $16 $4 $20 $17 $0 $0 $40 $10 $15 $20 $0 $1 $59 $241 $3 $1 $20 $2 $8 $6 $45 $75 $0 $16 $4 $9 $12 $5 $5 $7 $1 $1 $96 $11 $5 $4 $5 $1 $7 $5 $0 $9 $2 $1 $15 $5 $8 $20 $1 $9 $15 $2 $9 $0 $5 $10 $11 $7 $13 $2 $28 $3 $12 $30 $2 $4 $30 $35 $4 $6 $5 $0 $3 $20 $1 $13 $22 $3 $14 $15 $6 $1 $2 $11 $25 $12 $5 $2 $6 $2 $2 $2 $1 $6 $0 $4 $50 $40 $24 $6 $4 $1 $45 $95 $50 $1 $75 $15 $1 $9 $5 $8 $6 $5 $8 $4 $52 $11 $3 $10 $2 $9 $7 $5 $34 $0 $28 $0 $1 $19 $2 $3 $49 $9 $32 $5 $23 $45 $41 $2 $5 $8 $5 $0 $3 $26 $11 